# 03 — KV Cache Memory Model

Goal: calculate how KV cache grows with context length and active sequences. This explains why serving many users is mostly a memory-management problem.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

import pandas as pd

from vllm_lab.kv_cache import (
    COMMON_MODEL_SHAPES,
    bytes_to_gib,
    kv_cache_bytes_simplified,
    kv_cache_bytes_with_gqa,
    weight_memory_bytes,
)

Simplified formula: `KV bytes = 2 × layers × context_tokens × hidden_size × bytes_per_value × active_sequences`. The `2` is for key and value. This overestimates some grouped-query attention models, so we also include a GQA-aware formula.

In [ ]:
contexts = [512, 1024, 2048, 4096, 8192, 16384]
users = [1, 2, 4, 8, 16]
shape = COMMON_MODEL_SHAPES['llama_7b_like']

rows = []
for ctx in contexts:
    for u in users:
        b = kv_cache_bytes_simplified(shape.num_layers, shape.hidden_size, ctx, bytes_per_value=2, active_sequences=u)
        rows.append({'model_shape': shape.name, 'context_tokens': ctx, 'active_sequences': u, 'kv_cache_gib': bytes_to_gib(b)})

df = pd.DataFrame(rows)
df.pivot(index='context_tokens', columns='active_sequences', values='kv_cache_gib')

In [ ]:
ax = df[df['active_sequences'] == 1].plot(x='context_tokens', y='kv_cache_gib', marker='o', legend=False)
ax.set_title('KV cache grows linearly with context length')
ax.set_ylabel('KV cache GiB')

Now compare simplified full-attention estimate with grouped-query attention for a 70B-like model. GQA reduces KV cache by storing fewer KV heads than query heads.

In [ ]:
shape = COMMON_MODEL_SHAPES['llama_70b_like_gqa']
rows = []
for ctx in [4096, 8192, 16384, 32768]:
    simple = kv_cache_bytes_simplified(shape.num_layers, shape.hidden_size, ctx, 2, 1)
    gqa = kv_cache_bytes_with_gqa(shape.num_layers, shape.hidden_size, shape.num_heads, shape.num_kv_heads, ctx, 2, 1)
    rows.append({'context_tokens': ctx, 'simplified_gib': bytes_to_gib(simple), 'gqa_aware_gib': bytes_to_gib(gqa)})

pd.DataFrame(rows)

Weight memory is fixed after loading; KV cache grows with context and active sequences. That is why `max_model_len` and concurrency settings matter.

In [ ]:
models = [
    {'model': '7B', 'params_b': 7},
    {'model': '13B', 'params_b': 13},
    {'model': '70B', 'params_b': 70},
]
precisions = [('fp16', 2.0), ('int8', 1.0), ('int4_raw', 0.5)]
rows = []
for m in models:
    for name, bpp in precisions:
        rows.append({'model': m['model'], 'precision': name, 'weight_memory_gib': bytes_to_gib(weight_memory_bytes(m['params_b'] * 1e9, bpp))})

pd.DataFrame(rows).pivot(index='model', columns='precision', values='weight_memory_gib')